# Football Transfer Fee Prediction Pipeline

This notebook implements a clean, production-oriented version of the transfer fee prediction model. It includes data extraction, sophisticated feature engineering (including FIFA ratings integration), and model evaluation using both Linear Regression and Random Forest Regressor.

## 1. Environment Setup and Data Loading

In [3]:
!pip install fuzzywuzzy

Defaulting to user installation because normal site-packages is not writeable


In [4]:
import numpy as np
import pandas as pd
import os
import zipfile
import unicodedata
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from fuzzywuzzy import process

# Constants
TRANSFERS_PATH = '../data/transfers.zip'
RATINGS_PATH = '../data/ratings.zip'

def normalize_text(s):
    if not isinstance(s, str): return s
    return unicodedata.normalize('NFKD', s).encode('ascii', 'ignore').decode('utf-8')

transfers = pd.read_csv(TRANSFERS_PATH, compression='zip')
transfers.head()

C:\Users\thoma\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\fuzzywuzzy\fuzz.py:11: UserWarning: Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning
  warnings.warn('Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning')


,Name,Position,Age,Team_from,League_from,Team_to,League_to,Season,Market_value,Transfer_fee
0,Luís Figo,Right Winger,27,FC Barcelona,LaLiga,Real Madrid,LaLiga,2000-2001,NaN,60000000
1,Hernán Crespo,Centre-Forward,25,Parma,Serie A,Lazio,Serie A,2000-2001,NaN,56810000
2,Marc Overmars,Left Winger,27,Arsenal,Premier League,FC Barcelona,LaLiga,2000-2001,NaN,40000000
3,Gabriel Batistuta,Centre-Forward,31,Fiorentina,Serie A,AS Roma,Serie A,2000-2001,NaN,36150000
4,Nicolas Anelka,Centre-Forward,21,Real Madrid,LaLiga,Paris SG,Ligue 1,2000-2001,NaN,34500000


## 2. Initial Data Cleaning

We clean the seasons, normalize positions, and remove inconsistent rows (e.g., Age 0).

In [5]:
transfers['Season_transferred']=transfers['Season'].str.split('-').str[0]
transfers = transfers.astype({'Season_transferred':'int64'})
transfers = transfers.drop(columns =['Season'])
transfers.head()

,Name,Position,Age,Team_from,League_from,Team_to,League_to,Market_value,Transfer_fee,Season_transferred
0,Luís Figo,Right Winger,27,FC Barcelona,LaLiga,Real Madrid,LaLiga,NaN,60000000,2000
1,Hernán Crespo,Centre-Forward,25,Parma,Serie A,Lazio,Serie A,NaN,56810000,2000
2,Marc Overmars,Left Winger,27,Arsenal,Premier League,FC Barcelona,LaLiga,NaN,40000000,2000
3,Gabriel Batistuta,Centre-Forward,31,Fiorentina,Serie A,AS Roma,Serie A,NaN,36150000,2000
4,Nicolas Anelka,Centre-Forward,21,Real Madrid,LaLiga,Paris SG,Ligue 1,NaN,34500000,2000


In [6]:
print(transfers.Position.unique())
transfers.Position = transfers.Position.replace(to_replace=['Second Striker','Centre-Forward','Sweeper'],value = ['Forward','Forward','Defender'])
print(transfers.Position.unique())

<ArrowStringArray>
[      'Right Winger',     'Centre-Forward',        'Left Winger',
        'Centre-Back',   'Central Midfield', 'Attacking Midfield',
 'Defensive Midfield',     'Second Striker',         'Goalkeeper',
         'Right-Back',      'Left Midfield',          'Left-Back',
     'Right Midfield',            'Forward',            'Sweeper',
           'Defender',         'Midfielder']
Length: 17, dtype: str
<ArrowStringArray>
[      'Right Winger',            'Forward',        'Left Winger',
        'Centre-Back',   'Central Midfield', 'Attacking Midfield',
 'Defensive Midfield',         'Goalkeeper',         'Right-Back',
      'Left Midfield',          'Left-Back',     'Right Midfield',
           'Defender',         'Midfielder']
Length: 14, dtype: str


In [7]:
transfers.Age.unique()
transfers_weird = transfers.loc[transfers.Age == 0]
print(transfers_weird)

                  Name Position  Age Team_from    League_from  Team_to  \
236  Marzouq Al-Otaibi  Forward    0    Shabab   Saudi Arabia  Ittihad   

         League_to  Market_value  Transfer_fee  Season_transferred  
236   Saudi Arabia           NaN       2000000                2000  


In [8]:
transfers_midfield = transfers.loc[transfers.Position == 'Midfielder']
print(transfers_midfield.head(20))
transfers_defenders = transfers.loc[transfers.Position =='Defender']
print(transfers_defenders.head(20))

             Name    Position  Age Team_from     League_from Team_to  \
705  Tony Dinning  Midfielder   27    Wolves  First Division   Wigan   

                       League_to  Market_value  Transfer_fee  \
705  Second Division (bis 03/04)           NaN       1130000   

     Season_transferred  
705                2002  
                Name  Position  Age      Team_from     League_from  \
471    Thomas Hengen  Defender   26  VfL Wolfsburg    1.Bundesliga   
652  Sergio Hellings  Defender   17       Ajax U19  U19 Eredivisie   

            Team_to     League_to  Market_value  Transfer_fee  \
471  1.FC K'lautern  1.Bundesliga           NaN       2250000   
652  Anderlecht U19       Belgium           NaN       2000000   

     Season_transferred  
471                2001  
652                2002  


In [9]:
transfers_cleaned= transfers[~((transfers.Position=='Midfielder')|(transfers.Position=='Defender')|(transfers.Age ==0))]
transfers_cleaned['Transfer_fee_in_mln']=transfers_cleaned['Transfer_fee']/1000000
transfers_cleaned['Market_value_in_mln']=transfers_cleaned['Market_value']/1000000
transfers_cleaned = transfers_cleaned.drop(labels = ['Transfer_fee','Market_value'],axis = 1)
pd.to_datetime(transfers_cleaned['Season_transferred'],format ='%Y')
transfers_cleaned.head()

,Name,Position,Age,Team_from,League_from,Team_to,League_to,Season_transferred,Transfer_fee_in_mln,Market_value_in_mln
0,Luís Figo,Right Winger,27,FC Barcelona,LaLiga,Real Madrid,LaLiga,2000,60.00,NaN
1,Hernán Crespo,Forward,25,Parma,Serie A,Lazio,Serie A,2000,56.81,NaN
2,Marc Overmars,Left Winger,27,Arsenal,Premier League,FC Barcelona,LaLiga,2000,40.00,NaN
3,Gabriel Batistuta,Forward,31,Fiorentina,Serie A,AS Roma,Serie A,2000,36.15,NaN
4,Nicolas Anelka,Forward,21,Real Madrid,LaLiga,Paris SG,Ligue 1,2000,34.50,NaN


In [10]:
season_fee_stats = (
    transfers_cleaned
    .groupby('Season_transferred')['Transfer_fee_in_mln']
    .agg(mean_transfer_fee='mean', median_transfer_fee='median')
    .reset_index()
) #saving aggregate stats if called back later

## 3. Feature Engineering: Basic Features

Extracting Lastnames and normalizing text for reliable merging.

In [11]:
transfers_2015_18 = transfers_cleaned[transfers_cleaned['Season_transferred'] > 2014].copy()
transfers_2015_18['Lastname'] = transfers_2015_18['Name'].apply(lambda x: x.split(' ')[-1])

for col in ['Lastname', 'Team_from', 'Team_to']:
    transfers_2015_18[col] = transfers_2015_18[col].apply(normalize_text)

transfers_2015_18.head()

,Name,Position,Age,Team_from,League_from,Team_to,League_to,Season_transferred,Transfer_fee_in_mln,Market_value_in_mln,Lastname
3713,Kevin De Bruyne,Attacking Midfield,24,VfL Wolfsburg,1.Bundesliga,Man City,Premier League,2015,76.0,45.0,Bruyne
3714,Raheem Sterling,Right Winger,20,Liverpool,Premier League,Man City,Premier League,2015,63.7,40.0,Sterling
3715,Ángel Di María,Right Winger,27,Man Utd,Premier League,Paris SG,Ligue 1,2015,63.0,50.0,Maria
3716,Anthony Martial,Left Winger,19,Monaco,Ligue 1,Man Utd,Premier League,2015,60.0,8.0,Martial
3717,Alex Teixeira,Attacking Midfield,26,Shakhtar D.,Premier Liga,JS Suning,Super League,2015,50.0,30.0,Teixeira


In [12]:
import zipfile

ratings_zip_path = '../data/ratings.zip'
with zipfile.ZipFile(ratings_zip_path) as zf:
    fifa_full_15 = pd.read_csv(zf.open('players_15.csv'))
    fifa_full_16 = pd.read_csv(zf.open('players_16.csv'))
    fifa_full_17 = pd.read_csv(zf.open('players_17.csv'))
    fifa_full_18 = pd.read_csv(zf.open('players_18.csv'))

fifa_full_15['Season'] = 2015
fifa_full_16['Season'] = 2016
fifa_full_17['Season'] = 2017
fifa_full_18['Season'] = 2018

# Quick schema inspection
print("Shapes:")
print("2015:", fifa_full_15.shape, "2016:", fifa_full_16.shape, "2017:", fifa_full_17.shape, "2018:", fifa_full_18.shape)

print("\nColumn count by year:")
print(len(fifa_full_15.columns), len(fifa_full_16.columns), len(fifa_full_17.columns), len(fifa_full_18.columns))

print("\nDo all years have identical columns?",
      set(fifa_full_15.columns) == set(fifa_full_16.columns) == set(fifa_full_17.columns) == set(fifa_full_18.columns))

# Merge full datasets for broader inspection
fifas_full_merged = pd.concat([fifa_full_15, fifa_full_16, fifa_full_17, fifa_full_18], ignore_index=True, sort=False)

print("\nMerged full dataset shape:", fifas_full_merged.shape)
print("\nFirst 50 columns:")
print(fifas_full_merged.columns[:50].tolist())

fifas_full_merged.head()
fifas_full_merged.sample(5, random_state=42)

Shapes:
2015: (15465, 105) 2016: (14881, 105) 2017: (17009, 105) 2018: (17592, 105)

Column count by year:
105 105 105 105

Do all years have identical columns? True

Merged full dataset shape: (64947, 105)

First 50 columns:
['sofifa_id', 'player_url', 'short_name', 'long_name', 'age', 'dob', 'height_cm', 'weight_kg', 'nationality', 'club', 'overall', 'potential', 'value_eur', 'wage_eur', 'player_positions', 'preferred_foot', 'international_reputation', 'weak_foot', 'skill_moves', 'work_rate', 'body_type', 'real_face', 'release_clause_eur', 'player_tags', 'team_position', 'team_jersey_number', 'loaned_from', 'joined', 'contract_valid_until', 'nation_position', 'nation_jersey_number', 'pace', 'shooting', 'passing', 'dribbling', 'defending', 'physic', 'gk_diving', 'gk_handling', 'gk_kicking', 'gk_reflexes', 'gk_speed', 'gk_positioning', 'player_traits', 'attacking_crossing', 'attacking_finishing', 'attacking_heading_accuracy', 'attacking_short_passing', 'attacking_volleys', 'skill_dribb

,sofifa_id,player_url,short_name,long_name,age,dob,height_cm,weight_kg,nationality,club,...,ldm,cdm,rdm,rwb,lb,lcb,cb,rcb,rb,Season
52569,192890,https://sofifa.com/player/192890/erbin-trejo/1...,E. Trejo,Erbín Trejo,27,1990-06-03,171,69,Mexico,Querétaro,...,66+1,66+1,66+1,67+1,66+1,62+1,62+1,62+1,66+1,2018
35671,194730,https://sofifa.com/player/194730/juan-carlos-m...,Juan Carlos,Juan Carlos Martín Corral,28,1988-01-20,187,82,Spain,Elche CF,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2017
273,152554,https://sofifa.com/player/152554/gael-clichy/1...,G. Clichy,Gaël Clichy,28,1985-07-26,176,65,France,Manchester City,...,75+1,75+1,75+1,80+1,79+1,76+1,76+1,76+1,79+1,2015
16234,188942,https://sofifa.com/player/188942/victor-wanyam...,V. Wanyama,Victor Wanyama,24,1991-06-25,188,76,Kenya,Southampton,...,77+0,77+0,77+0,75+0,75+0,80+0,80+0,80+0,75+0,2016
20660,206018,https://sofifa.com/player/206018/igor-vetokele...,I. Vetokele,Igor Vetokele,23,1992-03-23,173,74,Angola,Charlton Athletic,...,45+0,45+0,45+0,50+0,47+0,42+0,42+0,42+0,47+0,2016


## 4. Advanced Feature Engineering: FIFA Ratings Integration

We load FIFA player ratings (Overall/Potential) and use multiple fallback strategies to recover missing metadata like Nationality.

In [13]:
nat_frames = []
with zipfile.ZipFile(RATINGS_PATH) as zf:
    for yr in [2015, 2016, 2017, 2018]:
        f = pd.read_csv(zf.open(f'players_{yr-2000}.csv'))
        nat_col = 'nationality_name' if 'nationality_name' in f.columns else ('nationality' if 'nationality' in f.columns else None)
        if nat_col is None: continue

        mini = f[['short_name', 'club', nat_col, 'overall', 'potential']].copy()
        mini['Lastname'] = mini['short_name'].str.split(' ').str[-1].apply(normalize_text)
        mini['club'] = mini['club'].apply(normalize_text)
        mini['Season_transferred'] = yr
        mini = mini.rename(columns={nat_col: 'Nationality'})
        nat_frames.append(mini)

fifa_nat = pd.concat(nat_frames, ignore_index=True)
fifa_nat['Nationality'] = fifa_nat['Nationality'].apply(normalize_text)

fifa_nat_best_club = fifa_nat.sort_values('overall', ascending=False).drop_duplicates(subset=['Lastname', 'club', 'Season_transferred'])
fifa_nat_best_name_season = fifa_nat.sort_values('overall', ascending=False).drop_duplicates(subset=['Lastname', 'Season_transferred'])

print(f'FIFA lookup rows (club-level): {len(fifa_nat_best_club)}')
print(f'FIFA lookup rows (lastname-season): {len(fifa_nat_best_name_season)}')

FIFA lookup rows (club-level): 63797
FIFA lookup rows (lastname-season): 46302


## 5. Dataset Merging and Fallback Recovery

We attempt to match transfers to FIFA data using Team_from, then Team_to, and finally a broader Lastname+Season search.

In [14]:
# Merge on Team_from
transfers_fe = transfers_2015_18.merge(
    fifa_nat_best_club.rename(columns={'club': 'Team_from', 'overall': 'overall_from', 'potential': 'potential_from'}),
    on=['Lastname', 'Team_from', 'Season_transferred'], how='left'
)

# Fallback on Team_to for nulls
null_mask = transfers_fe['overall_from'].isna()
to_lookup = fifa_nat_best_club.rename(columns={'club': 'Team_to', 'overall': 'overall_to', 'potential': 'potential_to', 'Nationality': 'Nat_to'})
transfers_fe = transfers_fe.merge(to_lookup, on=['Lastname', 'Team_to', 'Season_transferred'], how='left')

# Final broader fallback
transfers_fe['overall'] = transfers_fe['overall_from'].combine_first(transfers_fe['overall_to'])
transfers_fe['potential'] = transfers_fe['potential_from'].combine_first(transfers_fe['potential_to'])

final_df = transfers_fe.dropna(subset=['overall']).copy()
print(f"Nationality nulls after fallback: {final_df['Nationality'].isna().sum()} / {len(final_df)}")

Nationality nulls after fallback: 40 / 382


## 6. Engineered Indicators for Bias and Performance

Creating buckets for Age and Position, and purchasing power indicators for leagues and clubs.

In [15]:
#Purchasing power proxy
league_median = final_df.groupby('League_to')['Transfer_fee_in_mln'].median()
final_df['league_median_fee_to'] = final_df['League_to'].map(league_median)

# Age buckets
final_df['age_bucket'] = pd.cut(final_df['Age'], bins=[14, 20, 24, 27, 30, 50], labels=['<20', '20-23', '24-26', '27-29', '30+'])

# Position buckets
pos_map = {
    'Centre-Back': 'Defender', 'Right-Back': 'Defender', 'Left-Back': 'Defender',
    'Central Midfield': 'Midfielder', 'Attacking Midfield': 'Midfielder', 'Defensive Midfield': 'Midfielder',
    'Right Winger': 'Winger', 'Left Winger': 'Winger', 'Forward': 'Forward', 'Goalkeeper': 'Goalkeeper'
}
final_df['pos_group'] = final_df['Position'].map(pos_map).fillna('Other')

final_df.head()

,Name,Position,Age,Team_from,League_from,Team_to,League_to,Season_transferred,Transfer_fee_in_mln,Market_value_in_mln,...,potential_from,short_name_y,Nat_to,overall_to,potential_to,overall,potential,league_median_fee_to,age_bucket,pos_group
0,Kevin De Bruyne,Attacking Midfield,24,VfL Wolfsburg,1.Bundesliga,Man City,Premier League,2015,76.0,45.0,...,86.0,NaN,NaN,NaN,NaN,81.0,86.0,14.55,20-23,Midfielder
1,Raheem Sterling,Right Winger,20,Liverpool,Premier League,Man City,Premier League,2015,63.7,40.0,...,86.0,NaN,NaN,NaN,NaN,80.0,86.0,14.55,<20,Winger
5,Christian Benteke,Forward,24,Aston Villa,Premier League,Liverpool,Premier League,2015,46.5,20.0,...,85.0,NaN,NaN,NaN,NaN,80.0,85.0,14.55,20-23,Forward
6,Nicolás Otamendi,Centre-Back,27,Valencia CF,LaLiga,Man City,Premier League,2015,44.6,25.0,...,84.0,NaN,NaN,NaN,NaN,80.0,84.0,14.55,24-26,Defender
7,Julian Draxler,Central Midfield,21,FC Schalke 04,1.Bundesliga,VfL Wolfsburg,1.Bundesliga,2015,43.0,22.0,...,85.0,NaN,NaN,NaN,NaN,80.0,85.0,10.25,20-23,Midfielder


# Fairness Framework

In [20]:
#TODO Define Fairness Framework

## 7. Model Training: Linear Regression Baseline

In [16]:
# Drop name-specific identifiers
df_ml = final_df.drop(columns=['Name', 'Lastname', 'Team_from', 'Team_to', 'Nationality', 'overall_from', 'potential_from', 'overall_to', 'potential_to', 'Nat_to'])
df_ml = pd.get_dummies(df_ml)

X = df_ml.drop(columns=['Transfer_fee_in_mln'])
y = df_ml['Transfer_fee_in_mln']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=69)

lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred = lr.predict(X_test)

print(f"MSE: {mean_squared_error(y_test, y_pred):.2f}")
print(f"R2: {r2_score(y_test, y_pred):.2f}")

MSE: 117.06
R2: 0.65


In [17]:
# Baseline RF model (no engineered FIFA features, mirroring adjusted original approach)
transfers_baseline = transfers_cleaned.drop(columns=['Name', 'Market_value_in_mln']).copy()
transfers_baseline = pd.get_dummies(transfers_baseline)

y_baseline = transfers_baseline['Transfer_fee_in_mln']
X_baseline = transfers_baseline.drop(columns=['Transfer_fee_in_mln'])

X_train_base, X_test_base, y_train_base, y_test_base = train_test_split(
    X_baseline, y_baseline, random_state=69, test_size=0.3
)
print(X_train_base.shape)
print(X_test_base.shape)
print(y_train_base.shape)
print(y_test_base.shape)

rfr_baseline = RandomForestRegressor(n_estimators=270, max_features=650, random_state=169)
rfr_baseline.fit(X_train_base, y_train_base)
y_pred_rfr = rfr_baseline.predict(X_test_base)

print(f"MSE: {mean_squared_error(y_test_base, y_pred_rfr):.2f}")
print(f"R2: {r2_score(y_test_base, y_pred_rfr):.2f}")

(3287, 1090)
(1409, 1090)
(3287,)
(1409,)
MSE: 60.72
R2: 0.36


In [19]:
#TODO fairness metrics for the baseline

## 8. Improved Modeling: Random Forest Regressor

Using an ensemble method to capture non-linear relationships between ability, market value, and purchasing power.

In [18]:
rfr = RandomForestRegressor(n_estimators=270, max_features=650, random_state=169)
rfr.fit(X_train, y_train)
y_pred_rfr = rfr.predict(X_test)

print(f"MSE: {mean_squared_error(y_test, y_pred_rfr):.2f}")
print(f"R2: {r2_score(y_test, y_pred_rfr):.2f}")

MSE: 89.88
R2: 0.73


# SHAP/LIME 

# ICE/PDP Plots